# 05 - Merge Characters into Primary Table

Following source `generate_merged_primary_chars.py`:
Merge tessellation, building, enclosure, street, and node character files
into a single `primary_chars` table indexed by tessellation cell.

Includes CHM canopy features (`chm_mean`, `chm_std`, `chm_max`, `chm_cover`)
carried through from the building character layer.

In [1]:
import os
if os.path.basename(os.getcwd()).startswith("notebooks"):
    os.chdir(os.path.dirname(os.getcwd()))
print("Working dir:", os.getcwd())

import sys
sys.path.insert(0, "scripts")
from pipeline_config import load_city_config
load_city_config()

import geopandas as gpd
import pandas as pd
from pathlib import Path

OUTPUT_ROOT = os.environ.get("OUTPUT_ROOT", "outputs")
CHARS = Path(OUTPUT_ROOT) / "04_chars"
OUT = Path(OUTPUT_ROOT) / "05_primary_chars"
OUT.mkdir(parents=True, exist_ok=True)

# CHM canopy column names — must match CHM_COLS in notebooks 01 and 04
CHM_COLS = [c.strip() for c in os.environ.get("CHM_COLS", "chm_mean,chm_std,chm_max,chm_cover").split(",") if c.strip()]

# Informational only: CHM columns are already absent from buildings_chars.parquet
# when notebook 01 ran with INCLUDE_CHM=False, so USED_KEYS filtering below
# (`if c in merged.columns`) drops them automatically — nothing else to toggle here.
INCLUDE_CHM = os.environ.get("INCLUDE_CHM", "1").strip().lower() not in ("0", "false", "no")

Working dir: C:\urban-form\urban_taxonomy
[pipeline_config] City config: config\canada.env


In [2]:
tessellation = gpd.read_parquet(CHARS / "tessellations_chars.parquet")
buildings    = gpd.read_parquet(CHARS / "buildings_chars.parquet")
enclosures   = gpd.read_parquet(CHARS / "enclosures_chars.parquet")
streets      = gpd.read_parquet(CHARS / "streets_chars.parquet")
nodes_df     = gpd.read_parquet(CHARS / "nodes_chars.parquet")

print(f"Tess: {len(tessellation):,}, Bldg: {len(buildings):,}, Encl: {len(enclosures):,}")
print(f"Streets: {len(streets):,}, Nodes: {len(nodes_df):,}")

Tess: 93,234, Bldg: 84,142, Encl: 24,052
Streets: 12,534, Nodes: 14,114


## Merge all into primary_chars

In [3]:
# Source used_keys: the 59 morphometric characters that go into clustering
# (from source utils.py). Plus our Canadian additions and CHM canopy stats.
USED_KEYS = [
    # Building shape (1-8)
    "sdbAre", "sdbPer", "sdbCoA", "ssbCCo", "ssbCor", "ssbSqu",
    "ssbERI", "ssbElo",
    # Building context
    "stbOri", "mtbSWR", "libNCo", "ldbPWL", "ltcBuA", "mtbAli",
    "mtbNDi", "ltbIBD", "stbCeA", "stbSAl",
    # Connected structures (49-57)
    "mibCou", "mibAre", "mibLen", "mibElo", "mibERI", "mibCCo",
    "mibLAL", "mibFR", "mibSCo",
    # Centroid-corner (dropped in clustering but kept in primary)
    "ssbCCM", "ssbCCD",
    # Tessellation (9-13)
    "stcOri", "sdcLAL", "sdcAre", "sscCCo", "sscERI",
    "mtcWNe", "mdcAre", "ltcWRB", "sicCAR", "stcSAl",
    # Tessellation deviations (58)
    "micBAD",
    # Street (14-20, 37)
    "sdsLen", "sssLin", "ldsMSL", "ldsRea", "ldsAre",
    "sisBpM", "sdsSPW", "sdsSPO", "sdsSWD", "sdsAre",
    # Street deviation (59-ish)
    "misBAD",
    # Node (21, 27-30, 36, 38-48)
    "sddAre", "mtdDeg", "mtdMDi", "midRea", "midAre",
    "lcdMes", "ldsCDL", "lddNDe", "linWID",
    "linPDE", "linP3W", "linP4W", "lcnClo", "xcnSCl",
    # Node deviation (59)
    "midBAD",
    # Enclosure
    "ldkAre", "ldkPer", "lskCCo", "lskERI", "lskCWA",
    "ltkOri", "ltkWNB", "likWCe", "likWBB",
    # Canadian additions (not in source)
    "HEIGHT", "volume", "floor_count",
    # "BLDG_POP",  # population — commented out
    # CHM (LiDAR canopy height model) — canopy stats per building buffer
    *CHM_COLS,
]

# --- Merge all element chars ---
merged = pd.merge(
    tessellation.drop(columns=["geometry"]),
    buildings.drop(columns=[c for c in ["nodeID", "geometry", "nID"] if c in buildings.columns]),
    right_index=True, left_index=True, how="left",
)

merged = merged.merge(
    enclosures.drop(columns="geometry"),
    right_on="eID" if "eID" in enclosures.columns else enclosures.index.name,
    left_on="enclosure_index",
    how="left",
)

merged = merged.merge(streets.drop(columns="geometry"), on="nID", how="left")
merged = merged.merge(nodes_df.drop(columns="geometry"), on="nodeID", how="left")

# Drop join keys
drop_cols = [c for c in ["nID", "eID", "nodeID", "mm_len", "cdsbool",
             "node_start", "node_end", "enclosure_index", "barea"] if c in merged.columns]
merged = merged.drop(columns=drop_cols)
merged = merged.set_index(tessellation.index)

# *** KEY STEP: select only morphometric columns ***
primary = merged[[c for c in USED_KEYS if c in merged.columns]].copy()

print(f"Primary chars: {primary.shape[0]:,} rows x {primary.shape[1]} columns")
print(f"Columns: {list(primary.columns)}")
missing = [c for c in USED_KEYS if c not in merged.columns]
if missing:
    print(f"Missing (ok if not computed): {missing}")

Primary chars: 93,234 rows x 75 columns
Columns: ['sdbAre', 'sdbPer', 'sdbCoA', 'ssbCCo', 'ssbCor', 'ssbSqu', 'ssbERI', 'ssbElo', 'stbOri', 'mtbSWR', 'libNCo', 'ldbPWL', 'ltcBuA', 'mtbAli', 'mtbNDi', 'ltbIBD', 'stbCeA', 'stbSAl', 'mibCou', 'mibAre', 'mibLen', 'mibElo', 'mibERI', 'mibCCo', 'mibLAL', 'mibFR', 'mibSCo', 'ssbCCM', 'ssbCCD', 'stcOri', 'sdcLAL', 'sdcAre', 'sscCCo', 'sscERI', 'mtcWNe', 'mdcAre', 'sicCAR', 'stcSAl', 'micBAD', 'sdsLen', 'sssLin', 'ldsMSL', 'sisBpM', 'sdsSPW', 'sdsSPO', 'sdsSWD', 'sdsAre', 'misBAD', 'sddAre', 'mtdDeg', 'mtdMDi', 'midRea', 'midAre', 'lcdMes', 'ldsCDL', 'lddNDe', 'linWID', 'linPDE', 'linP3W', 'linP4W', 'lcnClo', 'xcnSCl', 'midBAD', 'ldkAre', 'ldkPer', 'lskCCo', 'lskERI', 'lskCWA', 'ltkOri', 'ltkWNB', 'likWCe', 'likWBB', 'HEIGHT', 'volume', 'floor_count']
Missing (ok if not computed): ['ltcWRB', 'ldsRea', 'ldsAre', 'chm_mean', 'chm_std', 'chm_max', 'chm_cover']


In [4]:
primary.to_parquet(OUT / "primary_chars.parquet")
print(f"Saved to {OUT / 'primary_chars.parquet'}")

Saved to outputs_canada\regions\4810\05_primary_chars\primary_chars.parquet
